# 10 — Silver Credit Card Balance

**Credit Risk Intelligence Platform** — Camada Silver

Este notebook transforma a tabela Bronze `credit_risk.bronze.credit_card_balance` em uma tabela Silver tratada, padronizada e preparada para análise e Machine Learning.

## Pipeline

```
credit_risk.bronze.credit_card_balance  →  credit_risk.silver.credit_card_balance
```

## Sobre a tabela credit_card_balance

A tabela `credit_card_balance` contém o histórico mensal de saldo de cartões de crédito anteriores do cliente no Home Credit. Cada registro representa o status mensal de um cartão, identificado por `SK_ID_PREV` no mês `MONTHS_BALANCE`.

## Transformações aplicadas

1. **Remoção de metadados Bronze** — colunas `_ingestion_timestamp` e `_source_file`
2. **Padronização de categorias** — `trim()` em `NAME_CONTRACT_STATUS`
3. **Preservação de NULLs numéricos** — 9 colunas com NULLs preservados (significado: sem atividade no mês)
4. **Flags de validação** — valores monetários negativos sinalizados (não removidos)
5. **Colunas de controle** — timestamp, versão, origem, hash
6. **Auditoria** — registro completo da transformação

## Regras

> A Bronze **NÃO é modificada**. Todas as transformações criam novas tabelas Silver.
> Nenhum registro é removido sem justificativa documentada.
> Nenhuma agregação por cliente é realizada — a tabela permanece no nível de registros originais.
> Valores negativos em colunas financeiras são **preservados** — podem representar saldo credor ou estornos.
> A coluna `AMT_RECIVABLE` mantém o nome original do dataset (typo do Home Credit).

In [0]:
# ============================================================================
# CÉLULA 1 — Configuração, Imports e Parâmetros
# ============================================================================
from pyspark.sql import functions as F, types as T, Window
from datetime import datetime, timezone
import uuid

# ----------------------------------------------------------------------------
# Parâmetros do pipeline
# ----------------------------------------------------------------------------
PIPELINE_VERSION = "silver_v1.0"
NOTEBOOK_NAME = "10_silver_credit_card_balance"
EXECUTION_ID = str(uuid.uuid4())
BATCH_ID = f"silver_cc_bal_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
EXECUTION_TIMESTAMP = datetime.now(timezone.utc)

# ----------------------------------------------------------------------------
# Tabelas de origem (Bronze) e destino (Silver)
# ----------------------------------------------------------------------------
BRONZE_TABLE = "credit_risk.bronze.credit_card_balance"
SILVER_TABLE = "credit_risk.silver.credit_card_balance"
AUDIT_TABLE = "credit_risk.silver.audit_transformation"

# Tabelas Silver para integridade referencial
SILVER_PREV_APP = "credit_risk.silver.previous_application"
SILVER_APP_TRAIN = "credit_risk.silver.application_train"
SILVER_APP_TEST = "credit_risk.silver.application_test"

# Colunas de metadados Bronze a remover na Silver
BRONZE_META_COLS = ["_ingestion_timestamp", "_source_file"]

# ----------------------------------------------------------------------------
# Criar schema Silver se não existir
# ----------------------------------------------------------------------------
spark.sql("CREATE SCHEMA IF NOT EXISTS credit_risk.silver")
print(f"Schema credit_risk.silver verificado/criado.")

# ----------------------------------------------------------------------------
# Dicionário para registrar transformações aplicadas (para auditoria)
# ----------------------------------------------------------------------------
TRANSFORMATION_LOG = []

def log_transform(table_name, step, description, records_affected=0):
    """Registra uma transformação aplicada para auditoria."""
    TRANSFORMATION_LOG.append({
        "table": table_name,
        "step": step,
        "description": description,
        "records_affected": records_affected,
    })

print(f"⏱️ Execution ID: {EXECUTION_ID}")
print(f"📦 Batch ID: {BATCH_ID}")
print(f"🔧 Pipeline Version: {PIPELINE_VERSION}")

In [0]:
# ============================================================================
# CÉLULA 2 — Leitura da Bronze e Inspeção do Schema
# ============================================================================
# Carrega o DataFrame Bronze (sem modificá-lo) e inspeciona o schema real.

df_cc_bronze = spark.table(BRONZE_TABLE)

bronze_row_count = df_cc_bronze.count()
bronze_col_count = len(df_cc_bronze.columns)

print("=" * 70)
print("INSPEÇÃO INICIAL — BRONZE")
print("=" * 70)
print(f"\n📊 {BRONZE_TABLE}")
print(f"   Registros: {bronze_row_count:,}")
print(f"   Colunas: {bronze_col_count}")

# Schema detalhado
sep = "─" * 70
print(f"\n{sep}")
print("SCHEMA — credit_card_balance (tipos e nullable)")
print(sep)
for field in df_cc_bronze.schema.fields:
    print(f"   {field.name:<35} {field.dataType.simpleString():<12} nullable={field.nullable}")

# Verificar colunas-chave esperadas
print(f"\n{sep}")
print("COLUNAS-CHAVE")
print(sep)
for c in ["SK_ID_PREV", "SK_ID_CURR", "MONTHS_BALANCE"]:
    if c in df_cc_bronze.columns:
        print(f"   ✅ {c}: presente")
    else:
        print(f"   ❌ {c}: AUSENTE")

print("\n✅ Leitura da Bronze concluída!")

In [0]:
# ============================================================================
# CÉLULA 3 — Data Quality Inicial (Bronze)
# ============================================================================
# Análise de completude (NULLs), valores distintos e estatísticas básicas.

sep = "─" * 70

# ----------------------------------------------------------------------------
# NULLs por coluna
# ----------------------------------------------------------------------------
null_exprs = [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_cc_bronze.columns]
null_row = df_cc_bronze.agg(*null_exprs).collect()[0]
null_pairs = [(c, null_row[c]) for c in df_cc_bronze.columns if null_row[c] and null_row[c] > 0]
null_pairs.sort(key=lambda x: x[1], reverse=True)

print(sep)
print(f"NULLs POR COLUNA — {BRONZE_TABLE} ({len(null_pairs)} cols com nulls)")
print(sep)
for c, n in null_pairs:
    pct = n / bronze_row_count * 100
    print(f"   {c:<35} {n:>10,}  ({pct:.2f}%)")
if not null_pairs:
    print("   Nenhuma coluna com NULLs")

# ----------------------------------------------------------------------------
# Valores distintos por coluna
# ----------------------------------------------------------------------------
print(f"\n{sep}")
print("VALORES DISTINCTOS")
print(sep)
for c in df_cc_bronze.columns:
    d = df_cc_bronze.select(c).distinct().count()
    print(f"   {c:<35} {d:>10,}")

# ----------------------------------------------------------------------------
# Estatísticas numéricas (colunas financeiras principais)
# ----------------------------------------------------------------------------
print(f"\n{sep}")
print("ESTATÍSTICAS NUMÉRICAS (min, max, mean, mediana, stddev)")
print(sep)
numeric_inspect = [
    "MONTHS_BALANCE", "AMT_BALANCE", "AMT_CREDIT_LIMIT_ACTUAL",
    "AMT_DRAWINGS_CURRENT", "AMT_PAYMENT_CURRENT", "AMT_PAYMENT_TOTAL_CURRENT",
    "AMT_RECEIVABLE_PRINCIPAL", "AMT_RECIVABLE", "AMT_TOTAL_RECEIVABLE",
    "CNT_DRAWINGS_CURRENT", "CNT_INSTALMENT_MATURE_CUM", "SK_DPD", "SK_DPD_DEF"
]
for c in numeric_inspect:
    if c in df_cc_bronze.columns:
        stats = df_cc_bronze.select(c).summary("min", "max", "mean", "50%", "stddev").collect()
        print(f"\n   {c}:")
        for s in stats:
            print(f"      {s['summary']:<10}: {s[c]}")

print("\n✅ Data Quality inicial concluída!")

In [0]:
# ============================================================================
# CÉLULA 4 — Validação de Identificadores e Duplicidades
# ============================================================================
# Valida chaves SK_ID_PREV, SK_ID_CURR, MONTHS_BALANCE e analisa duplicidades.

sep = "─" * 70
print("=" * 70)
print("VALIDAÇÃO DE IDENTIFICADORES E DUPLICIDADES")
print("=" * 70)

# ----------------------------------------------------------------------------
# Chaves individuais
# ----------------------------------------------------------------------------
for c in ["SK_ID_PREV", "SK_ID_CURR", "MONTHS_BALANCE"]:
    if c in df_cc_bronze.columns:
        null_cnt = df_cc_bronze.filter(F.col(c).isNull()).count()
        distinct_cnt = df_cc_bronze.select(c).distinct().count()
        print(f"\n   {c}:")
        print(f"      NULL: {null_cnt}")
        print(f"      Distinct: {distinct_cnt:,}")

# ----------------------------------------------------------------------------
# Duplicidades
# ----------------------------------------------------------------------------
print(f"\n{sep}")
print("ANÁLISE DE DUPLICIDADES")
print(sep)

# Duplicidade completa
full_dups = bronze_row_count - df_cc_bronze.dropDuplicates().count()
print(f"\n   Duplicidade completa: {full_dups} linhas totalmente duplicadas")

# Duplicidade por SK_ID_PREV
sk_prev_dups = bronze_row_count - df_cc_bronze.select("SK_ID_PREV").distinct().count()
print(f"   Duplicidade por SK_ID_PREV: {sk_prev_dups} (esperado: muitos — cada prev tem multiplos meses)")

# Duplicidade por (SK_ID_PREV + MONTHS_BALANCE) — chave lógica
composite_dups = bronze_row_count - df_cc_bronze.select("SK_ID_PREV", "MONTHS_BALANCE").distinct().count()
print(f"   Duplicidade por (SK_ID_PREV + MONTHS_BALANCE): {composite_dups}")
if composite_dups == 0:
    print("   → A combinação SK_ID_PREV + MONTHS_BALANCE é única — nenhum tratamento necessário")

# Duplicidade por (SK_ID_PREV + SK_ID_CURR + MONTHS_BALANCE)
triple_dups = bronze_row_count - df_cc_bronze.select("SK_ID_PREV", "SK_ID_CURR", "MONTHS_BALANCE").distinct().count()
print(f"   Duplicidade por (SK_ID_PREV + SK_ID_CURR + MONTHS_BALANCE): {triple_dups}")

# ----------------------------------------------------------------------------
# Distribuição de registros por SK_ID_PREV
# ----------------------------------------------------------------------------
print(f"\n{sep}")
print("DISTRIBUIÇÃO DE REGISTROS POR SK_ID_PREV")
print(sep)

per_prev_stats = df_cc_bronze.groupBy("SK_ID_PREV").count().select("count")
summary = per_prev_stats.summary("min", "max", "mean", "50%").collect()
for s in summary:
    print(f"   {s['summary']:<10}: {s['count']}")

print("\n✅ Validação de identificadores e duplicidades concluída!")

In [0]:
# ============================================================================
# CÉLULA 5 — Análise de MONTHS_BALANCE
# ============================================================================
# Inspeciona o campo temporal relativo MONTHS_BALANCE.
# Valores negativos representam meses no passado (0 = mês atual da aplicação).

sep = "─" * 70
print("=" * 70)
print("ANÁLISE DE MONTHS_BALANCE")
print("=" * 70)

# Estatísticas completas
mb_stats = df_cc_bronze.select("MONTHS_BALANCE").summary(
    "min", "max", "mean", "50%", "stddev"
).collect()
print(f"\n   Estatísticas:")
for s in mb_stats:
    print(f"      {s['summary']:<10}: {s['MONTHS_BALANCE']}")

# Verificar NULLs
mb_null = df_cc_bronze.filter(F.col("MONTHS_BALANCE").isNull()).count()
print(f"\n   NULLs: {mb_null}")

# Verificar valores não negativos
non_negative = df_cc_bronze.filter(F.col("MONTHS_BALANCE") >= 0).count()
print(f"   Valores >= 0: {non_negative} (esperado: 0 — todos negativos)")
if non_negative == 0:
    print("   → ✅ Todos os valores são negativos — consistente com a semântica do Home Credit")

# Distribuição por faixas
print(f"\n   Distribuição por faixas:")
bins = [(-96, -85), (-84, -73), (-72, -61), (-60, -49), (-48, -37), (-36, -25), (-24, -13), (-12, -1)]
for lo, hi in bins:
    cnt = df_cc_bronze.filter((F.col("MONTHS_BALANCE") >= lo) & (F.col("MONTHS_BALANCE") <= hi)).count()
    print(f"      {lo:>4} a {hi:>4}: {cnt:>12,} ({cnt/bronze_row_count*100:.2f}%)")

# Valores distintos
mb_distinct = df_cc_bronze.select("MONTHS_BALANCE").distinct().orderBy("MONTHS_BALANCE").collect()
print(f"\n   Valores distintos: {len(mb_distinct)} (de {mb_distinct[0]['MONTHS_BALANCE']} a {mb_distinct[-1]['MONTHS_BALANCE']})")

print("\n✅ Análise de MONTHS_BALANCE concluída!")

In [0]:
# ============================================================================
# CÉLULA 6 — Análise de Campos Categóricos
# ============================================================================
# Inspeciona NAME_CONTRACT_STATUS: valores, distribuição, NULLs, valores inesperados.

sep = "─" * 70
print("=" * 70)
print("ANÁLISE DE CAMPOS CATEGÓRICOS")
print("=" * 70)

# ----------------------------------------------------------------------------
# NAME_CONTRACT_STATUS
# ----------------------------------------------------------------------------
null_cnt = df_cc_bronze.filter(F.col("NAME_CONTRACT_STATUS").isNull()).count()
vals = df_cc_bronze.groupBy("NAME_CONTRACT_STATUS").count().orderBy(F.desc("count")).collect()

print(f"\n   NAME_CONTRACT_STATUS ({len(vals)} distinct, NULL={null_cnt}):")
print(f"   {'Valor':<30} {'Qtd':>12} {'%':>10}")
print(f"   {sep}")
for r in vals:
    pct = r['count'] / bronze_row_count * 100
    print(f"   {str(r['NAME_CONTRACT_STATUS']):<30} {r['count']:>12,} {pct:>9.2f}%")

# Verificar valores inesperados
expected_statuses = {
    "Active", "Completed", "Signed", "Demand",
    "Sent proposal", "Refused", "Approved"
}
actual_statuses = {r['NAME_CONTRACT_STATUS'] for r in vals}
unexpected = actual_statuses - expected_statuses
if unexpected:
    print(f"\n   ⚠️ Valores inesperados: {unexpected}")
else:
    print(f"\n   ✅ Todos os valores são conhecidos/esperados no Home Credit")

print("\n✅ Análise de campos categóricos concluída!")

In [0]:
# ============================================================================
# CÉLULA 7 — Análise de Campos Financeiros
# ============================================================================
# Inspeciona todas as colunas financeiras: saldos, limites, saques, pagamentos.
# Verifica negativos, zeros, NULLs, valores extremos.

sep = "─" * 70
print("=" * 70)
print("ANÁLISE DE CAMPOS FINANCEIROS")
print("=" * 70)

# Todas as colunas AMT_* e CNT_* e SK_DPD*
fin_cols = [c for c in df_cc_bronze.columns
            if c.startswith("AMT_") or c.startswith("CNT_") or c in ["SK_DPD", "SK_DPD_DEF"]]

for c in fin_cols:
    if c in df_cc_bronze.columns:
        null_cnt = df_cc_bronze.filter(F.col(c).isNull()).count()
        neg_cnt = df_cc_bronze.filter(F.col(c) < 0).count()
        zero_cnt = df_cc_bronze.filter(F.col(c) == 0).count()
        stats = df_cc_bronze.filter(F.col(c).isNotNull()).select(
            F.min(c).alias("min"), F.max(c).alias("max")).collect()[0]

        print(f"\n   {c}:")
        print(f"      NULL: {null_cnt:,} ({null_cnt/bronze_row_count*100:.2f}%)")
        print(f"      Negativos: {neg_cnt:,}")
        print(f"      Zeros: {zero_cnt:,} ({zero_cnt/bronze_row_count*100:.2f}%)")
        print(f"      Min: {stats['min']}  Max: {stats['max']}")
        if neg_cnt > 0:
            print(f"      ⚠️ {neg_cnt} valores negativos — preservados (podem representar saldo credor/estorno)")

print("\n✅ Análise de campos financeiros concluída!")

In [0]:
# ============================================================================
# CÉLULA 8 — Regras de Consistência
# ============================================================================
# Cria validações para colunas onde a semântica permite verificar consistência.
# Não transforma inconsistências em exclusões — apenas registra.

sep = "─" * 70
print("=" * 70)
print("REGRAS DE CONSISTÊNCIA")
print("=" * 70)

# ----------------------------------------------------------------------------
# Regras de não-negatividade (onde aplicável)
# ----------------------------------------------------------------------------
rules = [
    ("AMT_CREDIT_LIMIT_ACTUAL >= 0", df_cc_bronze.filter(F.col("AMT_CREDIT_LIMIT_ACTUAL") < 0).count()),
    ("CNT_DRAWINGS_CURRENT >= 0", df_cc_bronze.filter(F.col("CNT_DRAWINGS_CURRENT") < 0).count()),
    ("SK_DPD >= 0", df_cc_bronze.filter(F.col("SK_DPD") < 0).count()),
    ("SK_DPD_DEF >= 0", df_cc_bronze.filter(F.col("SK_DPD_DEF") < 0).count()),
    ("MONTHS_BALANCE in [-96, -1]", df_cc_bronze.filter((F.col("MONTHS_BALANCE") < -96) | (F.col("MONTHS_BALANCE") > -1)).count()),
]

print(f"\n   {'Regra':<40} {'Afetados':>10} {'%':>8} {'Status':>8}")
print(f"   {sep}")
for rule, affected in rules:
    pct = affected / bronze_row_count * 100 if bronze_row_count > 0 else 0
    status = "PASS" if affected == 0 else "WARNING"
    print(f"   {rule:<40} {affected:>10,} {pct:>7.2f}% {status:>8}")

# ----------------------------------------------------------------------------
# Valores negativos em colunas financeiras (WARNING — não são erros necessariamente)
# ----------------------------------------------------------------------------
print(f"\n{sep}")
print("VALORES NEGATIVOS EM COLUNAS FINANCEIRAS")
print(sep)
neg_rules = [
    ("AMT_BALANCE < 0", df_cc_bronze.filter(F.col("AMT_BALANCE") < 0).count()),
    ("AMT_DRAWINGS_ATM_CURRENT < 0", df_cc_bronze.filter(F.col("AMT_DRAWINGS_ATM_CURRENT") < 0).count()),
    ("AMT_DRAWINGS_CURRENT < 0", df_cc_bronze.filter(F.col("AMT_DRAWINGS_CURRENT") < 0).count()),
    ("AMT_RECEIVABLE_PRINCIPAL < 0", df_cc_bronze.filter(F.col("AMT_RECEIVABLE_PRINCIPAL") < 0).count()),
    ("AMT_RECIVABLE < 0", df_cc_bronze.filter(F.col("AMT_RECIVABLE") < 0).count()),
    ("AMT_TOTAL_RECEIVABLE < 0", df_cc_bronze.filter(F.col("AMT_TOTAL_RECEIVABLE") < 0).count()),
]
print(f"   {'Regra':<40} {'Afetados':>10} {'%':>8}")
print(f"   {sep}")
for rule, affected in neg_rules:
    pct = affected / bronze_row_count * 100
    print(f"   {rule:<40} {affected:>10,} {pct:>7.2f}%")

print(f"\n   ℹ️ Valores negativos em AMT_RECIVABLE e AMT_TOTAL_RECEIVABLE representam")
print(f"      saldo credor (cliente pagou mais do que devia) — preservados sem alteração")

print("\n✅ Regras de consistência validadas!")

In [0]:
# ============================================================================
# CÉLULA 9 — Integridade Referencial
# ============================================================================
# Valida SK_ID_PREV contra silver.previous_application e
# SK_ID_CURR contra silver.application_train + application_test.

sep = "─" * 70
print("=" * 70)
print("INTEGRIDADE REFERENCIAL")
print("=" * 70)

# ----------------------------------------------------------------------------
# SK_ID_PREV vs silver.previous_application
# ----------------------------------------------------------------------------
print(f"\n📊 credit_card_balance.SK_ID_PREV vs silver.previous_application:")
try:
    prev_app_sk = spark.table(SILVER_PREV_APP).select("SK_ID_PREV").distinct()
    cc_sk_prev = df_cc_bronze.select("SK_ID_PREV").distinct()
    cc_sk_prev_count = cc_sk_prev.count()

    matched_prev = cc_sk_prev.join(prev_app_sk, "SK_ID_PREV", "inner").count()
    unmatched_prev = cc_sk_prev_count - matched_prev

    print(f"   SK_ID_PREV distintos na credit_card: {cc_sk_prev_count:,}")
    print(f"   Correspondidos: {matched_prev:,} ({matched_prev/cc_sk_prev_count*100:.2f}%)")
    print(f"   Sem correspondência: {unmatched_prev:,} ({unmatched_prev/cc_sk_prev_count*100:.2f}%)")
    if unmatched_prev == 0:
        print(f"   → ✅ Todos os SK_ID_PREV têm correspondência em previous_application")
    else:
        print(f"   → ⚠️ {unmatched_prev} SK_ID_PREV sem correspondência (registros preservados)")
except Exception as e:
    print(f"   ⚠️ Não foi possível verificar: {e}")

# ----------------------------------------------------------------------------
# SK_ID_CURR vs silver.application_train + application_test
# ----------------------------------------------------------------------------
print(f"\n📊 credit_card_balance.SK_ID_CURR vs silver.application (train + test):")
try:
    app_train_sk = spark.table(SILVER_APP_TRAIN).select("SK_ID_CURR").distinct()
    app_test_sk = spark.table(SILVER_APP_TEST).select("SK_ID_CURR").distinct()
    all_app_sk = app_train_sk.union(app_test_sk).distinct()

    cc_sk_curr = df_cc_bronze.select("SK_ID_CURR").distinct()
    cc_sk_curr_count = cc_sk_curr.count()

    matched_curr = cc_sk_curr.join(all_app_sk, "SK_ID_CURR", "inner").count()
    unmatched_curr = cc_sk_curr_count - matched_curr

    print(f"   SK_ID_CURR distintos na credit_card: {cc_sk_curr_count:,}")
    print(f"   Correspondidos: {matched_curr:,} ({matched_curr/cc_sk_curr_count*100:.2f}%)")
    print(f"   Sem correspondência: {unmatched_curr:,} ({unmatched_curr/cc_sk_curr_count*100:.2f}%)")
    if unmatched_curr == 0:
        print(f"   → ✅ Todos os SK_ID_CURR têm correspondência em application")
    else:
        print(f"   → ⚠️ {unmatched_curr} SK_ID_CURR sem correspondência (registros preservados)")
except Exception as e:
    print(f"   ⚠️ Não foi possível verificar: {e}")

print("\n✅ Integridade referencial validada!")

In [0]:
# ============================================================================
# CÉLULA 10 — Funções de Transformação Reutilizáveis
# ============================================================================
# Funções modulares aplicadas na transformação Bronze → Silver.

def remove_bronze_metadata(df, table_name):
    """Remove colunas de metadados da Bronze."""
    cols_to_drop = [c for c in BRONZE_META_COLS if c in df.columns]
    if cols_to_drop:
        df = df.drop(*cols_to_drop)
        log_transform(table_name, "remove_metadata", f"Removidas colunas Bronze: {cols_to_drop}")
    return df


def standardize_categories(df, table_name):
    """Padroniza colunas categóricas: trim de espaços extras."""
    string_cols = [f.name for f in df.schema.fields if f.dataType.simpleString() == "string"]
    for col_name in string_cols:
        df = df.withColumn(col_name, F.trim(F.col(col_name)))
    log_transform(table_name, "standardize_categories",
                  f"Trim aplicado em {len(string_cols)} colunas string")
    print(f"   ✅ Padronização: trim aplicado em {len(string_cols)} colunas string")
    return df


def preserve_numeric_nulls(df, table_name, row_count):
    """Documenta NULLs em colunas numéricas.
    9 colunas têm NULLs significativos — preservados sem substituição.
    Significado: sem atividade de saque/pagamento/parcela no mês."""
    null_cols = [
        "AMT_DRAWINGS_ATM_CURRENT", "AMT_DRAWINGS_OTHER_CURRENT", "AMT_DRAWINGS_POS_CURRENT",
        "CNT_DRAWINGS_ATM_CURRENT", "CNT_DRAWINGS_OTHER_CURRENT", "CNT_DRAWINGS_POS_CURRENT",
        "AMT_INST_MIN_REGULARITY", "CNT_INSTALMENT_MATURE_CUM", "AMT_PAYMENT_CURRENT"
    ]
    for c in null_cols:
        if c in df.columns:
            null_cnt = df.filter(F.col(c).isNull()).count()
            if null_cnt > 0:
                log_transform(table_name, "preserve_numeric_null",
                    f"{c}: {null_cnt} NULLs preservados (sem atividade no mês)", null_cnt)
                print(f"   ℹ️ {c}: {null_cnt} NULLs preservados")
    return df


def add_validation_flags(df, table_name, row_count):
    """Adiciona flags de validação para valores negativos em colunas financeiras.
    Valores negativos são preservados — flags permitem filtragem posterior."""
    flags_added = []

    # AMT_BALANCE negativo (saldo credor/estorno)
    if "AMT_BALANCE" in df.columns:
        neg = df.filter(F.col("AMT_BALANCE") < 0).count()
        df = df.withColumn("FLAG_AMT_BALANCE_NEGATIVE",
            F.when(F.col("AMT_BALANCE") < 0, 1).otherwise(0))
        flags_added.append(("FLAG_AMT_BALANCE_NEGATIVE", neg))

    # AMT_DRAWINGS_CURRENT negativo (anomalia rara)
    if "AMT_DRAWINGS_CURRENT" in df.columns:
        neg = df.filter(F.col("AMT_DRAWINGS_CURRENT") < 0).count()
        df = df.withColumn("FLAG_AMT_DRAWINGS_NEGATIVE",
            F.when(F.col("AMT_DRAWINGS_CURRENT") < 0, 1).otherwise(0))
        flags_added.append(("FLAG_AMT_DRAWINGS_NEGATIVE", neg))

    # AMT_RECIVABLE negativo (saldo credor — comum)
    if "AMT_RECIVABLE" in df.columns:
        neg = df.filter(F.col("AMT_RECIVABLE") < 0).count()
        df = df.withColumn("FLAG_AMT_RECIVABLE_NEGATIVE",
            F.when(F.col("AMT_RECIVABLE") < 0, 1).otherwise(0))
        flags_added.append(("FLAG_AMT_RECIVABLE_NEGATIVE", neg))

    # AMT_TOTAL_RECEIVABLE negativo
    if "AMT_TOTAL_RECEIVABLE" in df.columns:
        neg = df.filter(F.col("AMT_TOTAL_RECEIVABLE") < 0).count()
        df = df.withColumn("FLAG_AMT_TOTAL_RECEIVABLE_NEGATIVE",
            F.when(F.col("AMT_TOTAL_RECEIVABLE") < 0, 1).otherwise(0))
        flags_added.append(("FLAG_AMT_TOTAL_RECEIVABLE_NEGATIVE", neg))

    # AMT_RECEIVABLE_PRINCIPAL negativo
    if "AMT_RECEIVABLE_PRINCIPAL" in df.columns:
        neg = df.filter(F.col("AMT_RECEIVABLE_PRINCIPAL") < 0).count()
        df = df.withColumn("FLAG_AMT_RECEIVABLE_PRINCIPAL_NEGATIVE",
            F.when(F.col("AMT_RECEIVABLE_PRINCIPAL") < 0, 1).otherwise(0))
        flags_added.append(("FLAG_AMT_RECEIVABLE_PRINCIPAL_NEGATIVE", neg))

    for flag_name, affected_count in flags_added:
        log_transform(table_name, "validation_flag",
            f"{flag_name}: {affected_count} registros marcados", affected_count)
        print(f"   ✅ {flag_name}: {affected_count} registros")
    return df


def add_control_columns(df, source_table):
    """Adiciona colunas de controle técnicas da Silver."""
    df = df.withColumn("silver_processing_timestamp", F.current_timestamp())
    df = df.withColumn("silver_processing_date", F.current_date())
    df = df.withColumn("silver_pipeline_version", F.lit(PIPELINE_VERSION))
    df = df.withColumn("source_table", F.lit(source_table))

    data_cols = [c for c in df.columns if c not in [
        "silver_processing_timestamp", "silver_processing_date",
        "silver_pipeline_version", "source_table"
    ]]
    hash_expr = F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit("NULL")) for c in data_cols])
    df = df.withColumn("record_hash", F.md5(hash_expr))
    print(f"   ✅ Colunas de controle adicionadas (timestamp, date, version, source, hash)")
    return df


def apply_silver_transformations(df, table_name, source_table, row_count):
    """Aplica todas as transformações Silver em sequência."""
    print(f"\n{'─' * 60}")
    print(f"🔧 Transformando: {table_name}")
    print(f"{'─' * 60}")

    df = remove_bronze_metadata(df, table_name)
    df = standardize_categories(df, table_name)
    df = preserve_numeric_nulls(df, table_name, row_count)
    df = add_validation_flags(df, table_name, row_count)
    df = add_control_columns(df, source_table)

    print(f"   ✅ Transformações concluídas para {table_name}")
    return df


print("✅ Funções de transformação definidas!")

In [0]:
# ============================================================================
# CÉLULA 11 — Execução das Transformações
# ============================================================================
# Aplica as transformações Silver na tabela credit_card_balance.
# O DataFrame Bronze original não é modificado.

EXEC_START = datetime.now(timezone.utc)

print("=" * 70)
print("TRANSFORMAÇÃO SILVER — credit_card_balance")
print("=" * 70)
transform_start = datetime.now(timezone.utc)

df_cc_silver = apply_silver_transformations(
    df_cc_bronze, SILVER_TABLE, BRONZE_TABLE, bronze_row_count
)

transform_end = datetime.now(timezone.utc)
transform_duration = (transform_end - transform_start).total_seconds()
silver_row_count = df_cc_silver.count()
silver_col_count = len(df_cc_silver.columns)

print(f"\n   Bronze: {bronze_row_count:,} rows x {bronze_col_count} cols")
print(f"   Silver: {silver_row_count:,} rows x {silver_col_count} cols")
print(f"   Duração: {transform_duration:.1f}s")

print(f"\n{'=' * 70}")
print(f"⏱️ Tempo total de transformação: {transform_duration:.1f}s")
print(f"{'=' * 70}")

In [0]:
# ============================================================================
# CÉLULA 12 — Escrita da Tabela Silver (Delta Lake)
# ============================================================================
# Grava a tabela Silver usando mode("overwrite") com overwriteSchema.

print("=" * 70)
print("GRAVAÇÃO DA TABELA SILVER")
print("=" * 70)

print(f"\n📊 Gravando {SILVER_TABLE}...")
write_start = datetime.now(timezone.utc)

df_cc_silver.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable(SILVER_TABLE)

write_end = datetime.now(timezone.utc)
write_duration = (write_end - write_start).total_seconds()
print(f"   ✅ {SILVER_TABLE} gravada em {write_duration:.1f}s")
print(f"      Registros: {silver_row_count:,} | Colunas: {silver_col_count}")

EXEC_END = datetime.now(timezone.utc)
TOTAL_DURATION = (EXEC_END - EXEC_START).total_seconds()

print(f"\n{'=' * 70}")
print("✅ TABELA SILVER GRAVADA COM SUCESSO!")
print(f"{'=' * 70}")

In [0]:
# ============================================================================
# CÉLULA 13 — Auditoria da Transformação
# ============================================================================
# Cria/atualiza a tabela credit_risk.silver.audit_transformation (append mode).
from pyspark.sql.types import (StructType, StructField, StringType,
    IntegerType, DoubleType, TimestampType, LongType)

audit_record = {
    "execution_timestamp": EXECUTION_TIMESTAMP,
    "execution_id": EXECUTION_ID,
    "batch_id": BATCH_ID,
    "source_table": BRONZE_TABLE,
    "target_table": SILVER_TABLE,
    "source_row_count": bronze_row_count,
    "target_row_count": silver_row_count,
    "records_inserted": silver_row_count,
    "records_removed": 0,
    "records_changed": silver_row_count,
    "processing_duration_seconds": float(TOTAL_DURATION),
    "pipeline_version": PIPELINE_VERSION,
    "execution_status": "SUCCESS",
    "error_message": "",
}

audit_schema = StructType([
    StructField("execution_timestamp", TimestampType(), True),
    StructField("execution_id", StringType(), True),
    StructField("batch_id", StringType(), True),
    StructField("source_table", StringType(), True),
    StructField("target_table", StringType(), True),
    StructField("source_row_count", LongType(), True),
    StructField("target_row_count", LongType(), True),
    StructField("records_inserted", LongType(), True),
    StructField("records_removed", IntegerType(), True),
    StructField("records_changed", LongType(), True),
    StructField("processing_duration_seconds", DoubleType(), True),
    StructField("pipeline_version", StringType(), True),
    StructField("execution_status", StringType(), True),
    StructField("error_message", StringType(), True),
])

audit_df = spark.createDataFrame([audit_record], schema=audit_schema)

print(f"📊 Persistindo auditoria em {AUDIT_TABLE}...")
audit_df.write.mode("append").format("delta").saveAsTable(AUDIT_TABLE)

print(f"✅ Auditoria registrada: 1 registro em {AUDIT_TABLE}")
print("\nRegistros de auditoria (últimos 10):")
display(spark.table(AUDIT_TABLE).orderBy(F.col("execution_timestamp").desc()).limit(10))

In [0]:
# ============================================================================
# CÉLULA 14 — Data Quality Pós-Transformação (Bronze vs Silver)
# ============================================================================
# Compara métricas de qualidade antes (Bronze) e depois (Silver).

def compute_dq_metrics(df, table_name):
    """Computa métricas de DQ: row_count, col_count, null_count, duplicate_count."""
    row_count = df.count()
    col_count = len(df.columns)
    null_exprs = [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)) for c in df.columns]
    total_nulls = df.agg(*null_exprs).collect()[0]
    null_sum = sum([total_nulls[i] for i in range(len(df.columns))])
    if "SK_ID_PREV" in df.columns and "MONTHS_BALANCE" in df.columns:
        dup_count = row_count - df.select("SK_ID_PREV", "MONTHS_BALANCE").distinct().count()
    else:
        dup_count = 0
    return {
        "table": table_name,
        "row_count": row_count,
        "col_count": col_count,
        "null_count": null_sum,
        "null_percentage": round(null_sum / (row_count * col_count) * 100, 2) if row_count > 0 else 0,
        "duplicate_count": dup_count,
    }

# Ler tabela Silver recém-criada
df_silver = spark.table(SILVER_TABLE)

sep = "─" * 75
print("=" * 70)
print("DATA QUALITY: BRONZE vs SILVER")
print("=" * 70)

bronze_m = compute_dq_metrics(df_cc_bronze, BRONZE_TABLE)
silver_m = compute_dq_metrics(df_silver, SILVER_TABLE)

print(f"\n📊 credit_card_balance")
print(f"{'Métrica':<30} {'Bronze':>15} {'Silver':>15} {'Delta':>15}")
print(sep)
for key in ["row_count", "col_count", "null_count", "null_percentage", "duplicate_count"]:
    b = bronze_m[key]
    s = silver_m[key]
    d = s - b
    print(f"{key:<30} {b:>15,} {s:>15,} {d:>+15,}")

# Verificações específicas
print(f"\n{sep}")
print("VERIFICAÇÕES ESPECÍFICAS")
print(sep)

# Colunas de controle presentes
control_cols = ["silver_processing_timestamp", "silver_processing_date",
                "silver_pipeline_version", "source_table", "record_hash"]
for c in control_cols:
    present = c in df_silver.columns
    print(f"   Coluna {c}: {'✅ presente' if present else '❌ ausente'}")

# Colunas Bronze removidas
print(f"\n   Colunas Bronze removidas:")
for c in BRONZE_META_COLS:
    present = c in df_silver.columns
    print(f"      {c}: {'❌ ainda presente' if present else '✅ removida'}")

# Flags de validação
validation_flags = [c for c in df_silver.columns if c.startswith("FLAG_")]
print(f"\n   Flags de validação ({len(validation_flags)}):")
for flag in validation_flags:
    count = df_silver.filter(F.col(flag) == 1).count()
    print(f"      {flag}: {count} registros")

# Chave composta (SK_ID_PREV + MONTHS_BALANCE) na Silver
silver_composite_dups = silver_row_count - df_silver.select("SK_ID_PREV", "MONTHS_BALANCE").distinct().count()
print(f"\n   (SK_ID_PREV + MONTHS_BALANCE) duplicatas na Silver: {silver_composite_dups} (esperado: 0)")

# MONTHS_BALANCE preservado
print(f"\n   MONTHS_BALANCE (Silver):")
mb_silver = df_silver.select("MONTHS_BALANCE").summary("min", "max").collect()
print(f"      Min: {mb_silver[0]['MONTHS_BALANCE']}  Max: {mb_silver[1]['MONTHS_BALANCE']}")

# NAME_CONTRACT_STATUS distribution preservada
print(f"\n   NAME_CONTRACT_STATUS (Silver):")
status_dist = df_silver.groupBy("NAME_CONTRACT_STATUS").count().orderBy(F.desc("count")).collect()
for r in status_dist:
    print(f"      {r['NAME_CONTRACT_STATUS']:<30} {r['count']:>12,}")

print("\n✅ Data Quality pós-transformação concluída!")

In [0]:
# ============================================================================
# CÉLULA 15 — Validação Final e Amostras
# ============================================================================
# Valida que a tabela Silver está correta e coerente com a Bronze.

sep = "─" * 70
print("=" * 70)
print("VALIDAÇÃO FINAL — TABELA SILVER")
print("=" * 70)

print(f"\n📊 {SILVER_TABLE}")
print(sep)

# Comparar row count
assert silver_row_count == bronze_row_count, \
    f"Row count mismatch: Bronze={bronze_row_count} vs Silver={silver_row_count}"
print(f"   ✅ Row count: {silver_row_count:,} (igual à Bronze)")

# Chave composta única
silver_dups = silver_row_count - df_silver.select("SK_ID_PREV", "MONTHS_BALANCE").distinct().count()
assert silver_dups == 0, f"Duplicatas encontradas: {silver_dups}"
print(f"   ✅ (SK_ID_PREV + MONTHS_BALANCE): único (0 duplicatas)")

# Colunas
print(f"   Colunas Bronze: {bronze_col_count}")
print(f"   Colunas Silver: {silver_col_count}")
print(f"   Colunas adicionadas: {silver_col_count - bronze_col_count}")
print(f"     - Removidas: 2 (metadados Bronze)")
print(f"     - Adicionadas: 5 flags validação + 5 colunas controle")

# Amostra
print(f"\n{sep}")
print(f"AMOSTRA — {SILVER_TABLE} (primeiras 20 linhas)")
print(sep)

sample_cols = [
    "SK_ID_PREV", "SK_ID_CURR", "MONTHS_BALANCE", "NAME_CONTRACT_STATUS",
    "AMT_BALANCE", "AMT_CREDIT_LIMIT_ACTUAL", "AMT_DRAWINGS_CURRENT",
    "AMT_PAYMENT_CURRENT", "AMT_PAYMENT_TOTAL_CURRENT",
    "AMT_RECIVABLE", "AMT_TOTAL_RECEIVABLE",
    "CNT_DRAWINGS_CURRENT", "CNT_INSTALMENT_MATURE_CUM",
    "SK_DPD", "SK_DPD_DEF",
    "FLAG_AMT_BALANCE_NEGATIVE", "FLAG_AMT_RECIVABLE_NEGATIVE",
    "silver_processing_timestamp", "silver_pipeline_version",
    "source_table", "record_hash"
]
sample_cols = [c for c in sample_cols if c in df_silver.columns]
display(df_silver.select(*sample_cols).limit(20))

# Estatísticas resumidas
print(f"\n{sep}")
print("ESTATÍSTICAS RESUMIDAS (Silver)")
print(sep)

silver_sk_prev = df_silver.select("SK_ID_PREV").distinct().count()
silver_sk_curr = df_silver.select("SK_ID_CURR").distinct().count()
print(f"   SK_ID_PREV distintos: {silver_sk_prev:,}")
print(f"   SK_ID_CURR distintos: {silver_sk_curr:,}")
print(f"   Registros por SK_ID_PREV (média): {silver_row_count / silver_sk_prev:.2f}")

print("\n✅ Validação final concluída com sucesso!")

In [0]:
# ============================================================================
# CÉLULA 16 — Resumo Final da Execução
# ============================================================================
# Exibe um resumo completo da transformação.

print("=" * 60)
print("SILVER CREDIT CARD BALANCE - RESUMO")
print("=" * 60)

print(f"\nOrigem:\n  {BRONZE_TABLE}")
print(f"\nDestino:\n  {SILVER_TABLE}")
print(f"\nRegistros Bronze:\n  {bronze_row_count:,}")
print(f"\nRegistros Silver:\n  {silver_row_count:,}")
print(f"\nRegistros removidos:\n  0")
print(f"\nRegistros alterados:\n  {silver_row_count:,} (transformações aplicadas)")
print(f"\nColunas:\n  Bronze: {bronze_col_count}")
print(f"  Silver: {silver_col_count}")

# NULLs preservados
nulls_preserved = sum(t["records_affected"] for t in TRANSFORMATION_LOG if t["step"] == "preserve_numeric_null")
print(f"\nNULLs preservados (numéricos):\n  {nulls_preserved:,}")

# Duplicidades
print(f"\nDuplicidades identificadas:\n  Completa: 0")
print(f"  (SK_ID_PREV + MONTHS_BALANCE): 0")
print(f"\nDuplicidades removidas:\n  0 (não havia duplicidades)")

# Anomalias identificadas
print(f"\nAnomalias identificadas:")
for t in TRANSFORMATION_LOG:
    if t["step"] == "validation_flag":
        print(f"  • {t['description']}")

# Transformações aplicadas
print(f"\nRegras aplicadas ({len(TRANSFORMATION_LOG)}):")
for t in TRANSFORMATION_LOG:
    print(f"  • {t['step']}: {t['description']}")

# Integridade referencial
print(f"\nIntegridade referencial:")
print(f"  SK_ID_PREV vs previous_application: verificada (registros preservados)")
print(f"  SK_ID_CURR vs application (train+test): verificada (registros preservados)")

# Warnings
warnings = [t for t in TRANSFORMATION_LOG if "WARNING" in t.get("description", "")]
print(f"\nWarnings:\n  {len(warnings)}")

print(f"\nStatus:\n  SUCCESS")
print(f"\nTempo:\n  {TOTAL_DURATION:.1f} segundos")
print(f"\n⏱️ Execution ID: {EXECUTION_ID}")
print(f"📦 Batch ID: {BATCH_ID}")
print(f"🔧 Pipeline: {PIPELINE_VERSION}")
print(f"\n{'=' * 60}")
print("✅ PIPELINE SILVER CREDIT CARD BALANCE CONCLUÍDO COM SUCESSO!")
print(f"{'=' * 60}")

## Transformações Aplicadas — Documentação

### 1. Remoção de metadados Bronze
Colunas `_ingestion_timestamp` e `_source_file` removidas (substituídas por colunas de controle Silver).

### 2. Padronização de categorias
- `trim()` aplicado em `NAME_CONTRACT_STATUS` (única coluna string de dados)
- Nenhum valor inesperado encontrado — todos os 7 valores são conhecidos no Home Credit

### 3. Tratamento de NULLs

9 colunas têm NULLs significativos, todos **preservados** sem substituição:

| Coluna | NULLs | % | Significado |
|--------|-------|---|------------|
| `AMT_DRAWINGS_ATM_CURRENT` | 749.816 | 19,52% | Sem saque ATM no mês |
| `AMT_DRAWINGS_OTHER_CURRENT` | 749.816 | 19,52% | Sem outros saques no mês |
| `AMT_DRAWINGS_POS_CURRENT` | 749.816 | 19,52% | Sem saque POS no mês |
| `CNT_DRAWINGS_ATM_CURRENT` | 749.816 | 19,52% | Sem saque ATM no mês |
| `CNT_DRAWINGS_OTHER_CURRENT` | 749.816 | 19,52% | Sem outros saques no mês |
| `CNT_DRAWINGS_POS_CURRENT` | 749.816 | 19,52% | Sem saque POS no mês |
| `AMT_INST_MIN_REGULARITY` | 305.236 | 7,95% | Sem parcela mínima definida |
| `CNT_INSTALMENT_MATURE_CUM` | 305.236 | 7,95% | Sem parcelas maturadas |
| `AMT_PAYMENT_CURRENT` | 767.988 | 20,00% | Sem pagamento no mês |

- Os 6 grupos de 749.816 NULLs (19,52%) correspondem aos mesmos registros — meses sem atividade de saque
- Substituir NULLs por zero alteraria a semântica (zero = atividade de valor zero, NULL = sem atividade)

### 4. Duplicidades
- 0 linhas totalmente duplicadas
- 0 duplicatas por (SK_ID_PREV + MONTHS_BALANCE) — chave composta única
- 0 duplicatas por (SK_ID_PREV + SK_ID_CURR + MONTHS_BALANCE)
- Nenhuma deduplicação foi necessária

### 5. MONTHS_BALANCE
- Faixa: -96 a -1 (96 valores distintos, todos negativos)
- Média: -34,52 | Mediana: -28 | Desvio: 26,67
- Valores negativos são **preservados** — representam meses no passado relativo à aplicação

### 6. NAME_CONTRACT_STATUS

| Valor | Quantidade | % |
|-------|-----------|---|
| Active | 3.698.436 | 96,31% |
| Completed | 128.918 | 3,36% |
| Signed | 11.058 | 0,29% |
| Demand | 1.365 | 0,04% |
| Sent proposal | 513 | 0,01% |
| Refused | 17 | 0,00% |
| Approved | 5 | 0,00% |

### 7. Valores negativos em colunas financeiras

| Coluna | Negativos | % | Interpretação |
|--------|-----------|---|---------------|
| `AMT_BALANCE` | 2.345 | 0,06% | Saldo credor/estorno |
| `AMT_DRAWINGS_ATM_CURRENT` | 1 | 0,00% | Anomalia (estorno) |
| `AMT_DRAWINGS_CURRENT` | 3 | 0,00% | Anomalia (estorno) |
| `AMT_RECEIVABLE_PRINCIPAL` | 2.428 | 0,06% | Pagamento excessivo |
| `AMT_RECIVABLE` | 109.338 | 2,85% | Saldo credor (comum) |
| `AMT_TOTAL_RECEIVABLE` | 109.330 | 2,85% | Saldo credor (comum) |

- Valores negativos em `AMT_RECIVABLE` e `AMT_TOTAL_RECEIVABLE` (~2,85%) representam saldo credor — cliente pagou mais do que devia
- Todos os valores negativos são **preservados** — flags de validação permitem filtragem posterior

### 8. Flags de validação criadas

| Flag | Descrição | Registros |
|------|-----------|-----------|
| `FLAG_AMT_BALANCE_NEGATIVE` | AMT_BALANCE < 0 | 2.345 |
| `FLAG_AMT_DRAWINGS_NEGATIVE` | AMT_DRAWINGS_CURRENT < 0 | 3 |
| `FLAG_AMT_RECIVABLE_NEGATIVE` | AMT_RECIVABLE < 0 | 109.338 |
| `FLAG_AMT_TOTAL_RECEIVABLE_NEGATIVE` | AMT_TOTAL_RECEIVABLE < 0 | 109.330 |
| `FLAG_AMT_RECEIVABLE_PRINCIPAL_NEGATIVE` | AMT_RECEIVABLE_PRINCIPAL < 0 | 2.428 |

### 9. Integridade referencial
- `credit_card_balance.SK_ID_PREV` → `silver.previous_application.SK_ID_PREV`
- `credit_card_balance.SK_ID_CURR` → `silver.application_train/test.SK_ID_CURR`
- Nenhum registro foi removido por falta de correspondência

### 10. Colunas de controle Silver
| Coluna | Tipo | Descrição |
|--------|------|------------|
| `silver_processing_timestamp` | timestamp | Momento da transformação |
| `silver_processing_date` | date | Data da transformação |
| `silver_pipeline_version` | string | Versão do pipeline (`silver_v1.0`) |
| `source_table` | string | Tabela de origem Bronze |
| `record_hash` | string | Hash MD5 de todos os campos para rastreabilidade |

### 11. Schema da tabela Silver
- **Bronze**: 25 colunas (23 dados + 2 metadados)
- **Silver**: 31 colunas (23 dados + 5 flags + 5 controle - 2 metadados removidos)
- Nenhuma coluna original foi modificada em tipo ou semântica
- Nota: `AMT_RECIVABLE` mantém o nome original do dataset (typo do Home Credit — não corrigido para preservar compatibilidade)